In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

print("OPENAI:", os.getenv("OPENAI_API_KEY") is not None)
print("LANGSMITH:", os.getenv("LANGSMITH_API_KEY") is not None)
print("TRACING:", os.getenv("LANGSMITH_TRACING"))
print("PROJECT:", os.getenv("LANGSMITH_PROJECT"))
print("ENDPOINT:", os.getenv("LANGSMITH_ENDPOINT"))

OPENAI: True
LANGSMITH: True
TRACING: true
PROJECT: openai-project
ENDPOINT: https://api.smith.langchain.com


In [2]:
from langchain_openai import ChatOpenAI

llm=ChatOpenAI(model='gpt-4o')

In [4]:
from langchain_community.document_loaders import WebBaseLoader


In [20]:
loader=WebBaseLoader("https://huggingface.co/docs/transformers/index")
loader

In [21]:
docs=loader.load()
docs

[Document(metadata={'source': 'https://huggingface.co/docs/transformers/index', 'title': 'Transformers · Hugging Face', 'description': 'We’re on a journey to advance and democratize artificial intelligence through open source and open science.', 'language': 'No language found.'}, page_content='\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nTransformers · Hugging Face\n\n\n\n\n\n\n    Hugging Face        Models  Datasets  Spaces  Buckets new Docs  Enterprise  Pricing      Log In Sign Up       Transformers documentation Transformers    Transformers  🏡 View all docsAWS Trainium & InferentiaAccelerateArgillaAutoTrainBitsandbytesCLIChat UIDataset viewerDatasetsDeploying on AWSDiffusersDistilabelEvaluateGoogle CloudGoogle TPUsGradioHubHub Python LibraryHuggingface.jsInference Endpoints (dedicated)Inference ProvidersKernelsLeRobotLeaderboardsLightevalMicrosoft AzureOptimumPEFTReachy MiniSafetensorsSentence TransformersTRLTasksText Embeddings InferenceText Generation InferenceTokenizers

In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents=text_splitter.split_documents(docs)
documents

[Document(metadata={'source': 'https://huggingface.co/docs/transformers/index', 'title': 'Transformers · Hugging Face', 'description': 'We’re on a journey to advance and democratize artificial intelligence through open source and open science.', 'language': 'No language found.'}, page_content='Transformers · Hugging Face'),
 Document(metadata={'source': 'https://huggingface.co/docs/transformers/index', 'title': 'Transformers · Hugging Face', 'description': 'We’re on a journey to advance and democratize artificial intelligence through open source and open science.', 'language': 'No language found.'}, page_content='Hugging Face        Models  Datasets  Spaces  Buckets new Docs  Enterprise  Pricing      Log In Sign Up       Transformers documentation Transformers    Transformers  🏡 View all docsAWS Trainium & InferentiaAccelerateArgillaAutoTrainBitsandbytesCLIChat UIDataset viewerDatasetsDeploying on AWSDiffusersDistilabelEvaluateGoogle CloudGoogle TPUsGradioHubHub Python LibraryHuggingfa

In [23]:
from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings()

In [24]:
from langchain_community.vectorstores import FAISS
vectorstoredb=FAISS.from_documents(documents,embeddings)
vectorstoredb

In [26]:
query='Transformers is designed for developers and machine learning engineers and researchers.'
result=vectorstoredb.similarity_search(query)
result[0].page_content

'for training and distributed training for PyTorch models. generate: Fast text generation with large language models (LLMs) and vision language models (VLMs), including support for streaming and multiple decoding strategies.  Design Read our Philosophy to learn more about Transformers’ design principles. Transformers is designed for developers and machine learning engineers and researchers. Its main design principles are: Fast and easy to use: Every model is implemented from only three main classes (configuration, model, and preprocessor) and can be quickly used for inference or training with Pipeline or Trainer. Pretrained models: Reduce your carbon footprint, compute cost and time by using a pretrained model instead of training an entirely new one. Each pretrained model is reproduced as closely as possible to the original model and offers state-of-the-art performance.   Learn If you’re new to Transformers or want to learn more about transformer models, we recommend starting with the'

In [28]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model='gpt-4o')

In [32]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the provided context.

Context:
{context}

Question:
{input}
""")

document_chain = create_stuff_documents_chain(llm, prompt)

document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based only on the provided context.\n\nContext:\n{context}\n\nQuestion:\n{input}\n'), additional_kwargs={})])
| ChatOpenAI(profile={'name': 'GPT-4o', 'release_date': '2024-05-13', 'last_updated': '2024-08-06', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attach

In [33]:
from langchain_core.documents import Document

document_chain.invoke({
    "input":"Transformers is designed for developers and machine learning engineers and researchers",
    "context":[Document(page_content="Transformers is designed for developers and machine learning engineers and researchers. Its main design principles are:Fast and easy to use: Every model is implemented from only three main classes (configuration, model, and preprocessor) and can be quickly used for inference or training with Pipeline or Trainer.Pretrained models: Reduce your carbon footprint, compute cost and time by using a pretrained model instead of training an entirely new one. Each pretrained model is reproduced as closely as possible to the original model and offers state-of-the-art performance.")]
})

'Yes, Transformers is designed for developers, machine learning engineers, and researchers.'

In [36]:
retriever=vectorstoredb.as_retriever()
from langchain_classic.chains import create_retrieval_chain
retriever_chain=create_retrieval_chain(retriever, document_chain)
retriever_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001DC9882E050>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based only on the provided context.\n\nContext:\n{context}\n\nQuestion:\n{input}\n'), additional_kwargs={})])

In [39]:
response= retriever_chain.invoke(
    {
        "input":"Transformers is designed for developers and machine learning engineers and researchers"
    }
)
print(response["answer"])
print(response["context"])

True. Transformers is indeed designed for developers, machine learning engineers, and researchers.
[Document(id='f43414e5-f58f-4a39-ba5d-b40faed0de37', metadata={'source': 'https://huggingface.co/docs/transformers/index', 'title': 'Transformers · Hugging Face', 'description': 'We’re on a journey to advance and democratize artificial intelligence through open source and open science.', 'language': 'No language found.'}, page_content='for training and distributed training for PyTorch models. generate: Fast text generation with large language models (LLMs) and vision language models (VLMs), including support for streaming and multiple decoding strategies.  Design Read our Philosophy to learn more about Transformers’ design principles. Transformers is designed for developers and machine learning engineers and researchers. Its main design principles are: Fast and easy to use: Every model is implemented from only three main classes (configuration, model, and preprocessor) and can be quickly 